# 05 - Baselines (k-means on the price only)

Clusters the days with only the mean and standard deviation of the log real price, without scalograms or VAE-LSTM.

Pipeline: mean and std of the log price (of the day, its backward 7-day mean or the week) -> k-means -> regime

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt
from IPython.display import display
from sklearn.manifold import TSNE

alt.data_transformers.enable('vegafusion')

from src.data import load_context, date_split, val_start, test_start
from src.clustering import with_level, level_columns
from src.plots import regime_timeline
from src.runs import prepare_runs, cluster_runs, tsne_grid, regime_strips
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

Set `level_window` and `tag` together: 1 and `""` for the daily baseline, 7 and `"7d"` for the 7-day baseline. For the calendar week baseline use `scalogram_set = "real_price_log_calweek"`, `study = "calweek"`, `level_window = 1` and `tag = ""`.
- `k_values` is always 3, 4 and 5.

In [ ]:
scalogram_set = "real_price_log_calweek"
k_values = [3, 4, 5]
study = "calweek"
level_window = 1
tag = ""
seeds = [42, 43, 44]

base_config = {
    "model": "level_only",
    "study": study,
    "scalogram_set": scalogram_set,
    "k_values": k_values,
    "features": "weekly mean and std of the log real price",
    "level_window": level_window,
    "val_start": val_start,
    "test_start": test_start,
}

## 1. Data and features

In [ ]:
scalogram_dir = Path("../data/processed/scalograms") / base_config["scalogram_set"]
metadata = pd.read_csv(scalogram_dir / "scalogram_metadata.csv", parse_dates=["start_datetime", "end_datetime"])
context = load_context(Path("../data/processed"), metadata)
train_idx, val_idx, test_idx = date_split(context["date"], base_config["val_start"], base_config["test_start"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]
print(exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx}))

level_cols = list(level_columns(metadata))
level = metadata.copy()
level[level_cols] = metadata[level_cols].rolling(base_config["level_window"], min_periods=1).mean()
features = with_level(np.zeros((len(metadata), 0)), level, train_idx)
print(f"Columns {level_cols}, backward mean over {base_config['level_window']} day(s), z-scored on training days")

## 2. Clustering, one pass per seed

In [ ]:
all_results = {}
for seed in seeds:
    seed_tag = "_".join(part for part in (tag, f"seed{seed}") if part)
    config, run_dirs, run_configs, _ = prepare_runs({**base_config, "tag": seed_tag, "seed": seed})
    for k, run_dir in run_dirs.items():
        print(f"seed={seed}, k={k}: {exp.run_id(run_dir)}")
    rng = np.random.default_rng(seed)
    plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
    coords = TSNE(n_components=2, perplexity=30, random_state=seed).fit_transform(features[plot_idx])
    results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, coords, plot_idx)
    all_results[seed] = (coords, plot_idx, results)

## 3. Regimes

In [ ]:
coords, plot_idx, results = all_results[seeds[0]]
display(tsne_grid(coords, plot_idx, results))
for k, ctx in results.items():
    display(regime_timeline(ctx, split_dates, title=f"Detected regimes over time (k={k})"))
display(regime_strips(results))